In [ ]:
import kagglehub



# Download latest version

path = kagglehub.dataset_download("mohammad2012191/q3-ka-ai-2026")



print("Path to dataset files:", path)


In [ ]:
import kagglehub
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
from torch.optim import AdamW
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader, TensorDataset
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

In [ ]:
# Task 1: Write your code here:
df = pd.read_csv(path + "/Q3_data.csv")

In [ ]:
# Task 2: Write your code here:
df.head()

In [ ]:
# Task 3: Write your code here:
df.info()

In [ ]:
# Task 4: Write your code here:
df.describe()

In [ ]:
# df = df.drop(columns=['D_87', 'D_88', 'D_110', 'B_39', 'D_111', 'D_108', 'B_42', 'D_73', 'D_138', 'D_136'])

In [ ]:
# Task 1: Write your code here:
df = df.fillna(0)
df.isna().sum().sort_values(ascending=False)

In [ ]:
# Task 2: Write your code here:
print(df.duplicated().sum())
df = df.drop_duplicates()
print(df.duplicated().sum())

In [ ]:
# Task 3: Write your code here:
#all features are numbers, encoding not needed

In [ ]:
# Task 4: Write your code here:
scaler = StandardScaler()
df[df.drop(columns='Target').columns] = scaler.fit_transform(df[df.drop(columns='Target').columns])
df.head()

In [ ]:
# Task 5: Write your code here:

print(f"Default: {df['Target'].sum()}")
print(f"No Default: {(df['Target'] == 0).sum()}")

# >>>>>>>>>>>>>>>>> the target is imbalanced <<<<<<<<<<<


In [ ]:
# %pip install kagglehub catboost xgboost tqdm imbalanced-learn -q
from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score, f1_score

In [ ]:
# Task 1: Write your code here:
X = df.drop(columns='Target').astype(float)
y = df['Target'].astype(float)


In [ ]:
# Task 2,3,4,5: Write your code here:
n = 8
skf = StratifiedKFold(n_splits=n, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)
f1_scores = []
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred, average='macro')

    f1_scores.append(f1)

print('Training done ^^')


print("averaged f1 score across all folds", np.mean(f1_scores))


In [ ]:
# Task 1: Write your code here:
feature_importance = pd.DataFrame({
    'feature': df.drop(columns='Target').columns,
    'importance': model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 50))
plt.barh(feature_importance['feature'], feature_importance['importance'])
plt.xlabel('Importance')
plt.title('Feature Importance')
plt.gca().invert_yaxis()
plt.show()

In [ ]:
# Task 2: Write your code here:
print('the golden feature is',  'P_2')

In [ ]:
# Task Bonus: Write your code here:


X = df[['P_2']].astype(float)
y = df['Target'].astype(float)

n = 8
skf = StratifiedKFold(n_splits=n, shuffle=True, random_state=42)

model = CatBoostClassifier(verbose=0, n_estimators=200, max_depth=4)
f1_scores = []
for fold_idx, (train_index, test_index) in enumerate(skf.split(X, y)):

    X_train, X_test = X.iloc[train_index], X.iloc[test_index]
    y_train, y_test = y.iloc[train_index], y.iloc[test_index]

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    f1 = f1_score(y_test, y_pred, average='macro')

    f1_scores.append(f1)

print('Training done ^^')
print("averaged f1 score across all folds", np.mean(f1_scores))

In [ ]:
# we astonomically reduced dimensenality and yet the f1 score difference is still good, we did not lose much